# YOLOv8 PPE Detection Training
## CVH-ppe-prueba1 Dataset on Google Colab

This notebook trains a YOLOv8 model for Personal Protective Equipment (PPE) detection using the CVH-ppe-prueba1 dataset.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install Required Libraries

In [ ]:
!pip install -q ultralytics opencv-python pillow matplotlib

## 3. Unzip Dataset

In [ ]:
import zipfile
import os
from pathlib import Path

# Set paths
gdrive_path = '/content/drive/MyDrive/Colab_Notebooks/data'
zip_path = os.path.join(gdrive_path, 'CVH-ppe-prueba1.v1i.yolov8.zip')
extract_path = '/content/dataset'

# Create extraction directory
os.makedirs(extract_path, exist_ok=True)

# Unzip the dataset
print(f"Unzipping {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Dataset extracted to {extract_path}")

## 4. Explore Dataset Structure

In [ ]:
import yaml

# Check extracted directory structure
print("Directory structure:")
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:3]:  # Show first 3 files
        print(f'{subindent}{file}')
    if len(files) > 3:
        print(f'{subindent}... and {len(files)-3} more files')

# Read and display data.yaml
yaml_path = os.path.join(extract_path, 'data.yaml')
if os.path.exists(yaml_path):
    print("\n--- Data Configuration (data.yaml) ---")
    with open(yaml_path, 'r') as f:
        data_config = yaml.safe_load(f)
        print(yaml.dump(data_config, default_flow_style=False))

## 5. Configure YOLOv8 Training Parameters

In [ ]:
# Training parameters
TRAINING_PARAMS = {
    'model': 'yolov8m.pt',  # yolov8n, yolov8s, yolov8m, yolov8l, yolov8x
    'epochs': 100,
    'batch_size': 16,
    'image_size': 640,
    'patience': 20,  # Early stopping patience
    'device': 0,  # GPU device (0 for first GPU)
    'confidence': 0.5,  # Confidence threshold
    'save': True,
    'project': '/content/runs/detect',
    'name': 'ppe_detection',
    'imgsz': 640,
    'workers': 4
}

print("Training Parameters:")
for key, value in TRAINING_PARAMS.items():
    print(f"  {key}: {value}")

## 6. Train YOLOv8 Model

**Note**: This will take time depending on your GPU and the selected model size. Monitor the training progress below.

In [ ]:
from ultralytics import YOLO

# Update data.yaml paths to be absolute
yaml_path = os.path.join(extract_path, 'data.yaml')
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# Update paths to absolute paths
data_config['path'] = extract_path
data_config['train'] = os.path.join(extract_path, 'train/images')
data_config['val'] = os.path.join(extract_path, 'valid/images')
data_config['test'] = os.path.join(extract_path, 'test/images')

# Save updated yaml
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)

# Load a YOLOv8 model
model = YOLO(TRAINING_PARAMS['model'])

# Train the model
results = model.train(
    data=yaml_path,
    epochs=TRAINING_PARAMS['epochs'],
    imgsz=TRAINING_PARAMS['image_size'],
    batch=TRAINING_PARAMS['batch_size'],
    patience=TRAINING_PARAMS['patience'],
    device=TRAINING_PARAMS['device'],
    project=TRAINING_PARAMS['project'],
    name=TRAINING_PARAMS['name'],
    verbose=True,
    save=TRAINING_PARAMS['save']
)

print("Training completed successfully!")

## 7. Validate Model Performance

In [ ]:
# Validate the model
best_model_path = os.path.join(TRAINING_PARAMS['project'], TRAINING_PARAMS['name'], 'weights', 'best.pt')
model = YOLO(best_model_path)

print("Validating model on test set...")
metrics = model.val()

print("\n--- Validation Metrics ---")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

# Show detailed metrics
print("\nDetailed results:")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")
print(f"Speed: {metrics.speed['inference']:.2f}ms per image")

## 8. Run Inference on Test Images

In [ ]:
import glob
from PIL import Image
import cv2

# Run inference on test images
test_images_path = os.path.join(extract_path, 'test/images')
test_images = glob.glob(os.path.join(test_images_path, '*.jpg'))

print(f"Found {len(test_images)} test images")

# Run predictions
results = model.predict(
    source=test_images[:5],  # Predict on first 5 test images
    conf=TRAINING_PARAMS['confidence'],
    imgsz=TRAINING_PARAMS['image_size'],
    save=True,
    save_txt=True,
    project='/content/inference_results',
    name='predictions'
)

print(f"Inference completed on {len(results)} images")
print(f"Results saved to /content/inference_results")

## 9. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import Image, display

# Display training metrics plots
metric_images = glob.glob(os.path.join(TRAINING_PARAMS['project'], TRAINING_PARAMS['name'], '*.png'))

if metric_images:
    print("Training Results:")
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for idx, img_path in enumerate(metric_images[:4]):
        img = mpimg.imread(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(os.path.basename(img_path))
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()

# Display inference results
inference_results = glob.glob(os.path.join('/content/inference_results/predictions', '*.jpg'))
print(f"\n\nInference Results on Test Images ({len(inference_results)} images):")

if inference_results:
    fig, axes = plt.subplots(1, min(3, len(inference_results)), figsize=(15, 5))
    if len(inference_results) == 1:
        axes = [axes]

    for idx, img_path in enumerate(inference_results[:3]):
        img = mpimg.imread(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(f"Prediction {idx+1}")
        axes[idx].axis('off')

    plt.tight_layout()
    plt.show()

## 10. Save and Download Results

In [ ]:
import shutil
from datetime import datetime

# Create timestamp for backup
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Copy results to Google Drive
results_backup_path = os.path.join(gdrive_path, f'yolov8_results_{timestamp}')
os.makedirs(results_backup_path, exist_ok=True)

# Copy trained model
shutil.copy(
    os.path.join(TRAINING_PARAMS['project'], TRAINING_PARAMS['name'], 'weights', 'best.pt'),
    os.path.join(results_backup_path, 'best.pt')
)

# Copy inference results
shutil.copytree(
    '/content/inference_results/predictions',
    os.path.join(results_backup_path, 'inference_results'),
    dirs_exist_ok=True
)

# Save summary
summary_path = os.path.join(results_backup_path, 'training_summary.txt')
with open(summary_path, 'w') as f:
    f.write("YOLOv8 PPE Detection Training Summary\n")
    f.write("=" * 50 + "\n\n")
    f.write(f"Dataset: CVH-ppe-prueba1.v1i.yolov8\n")
    f.write(f"Model: {TRAINING_PARAMS['model']}\n")
    f.write(f"Epochs: {TRAINING_PARAMS['epochs']}\n")
    f.write(f"Batch Size: {TRAINING_PARAMS['batch_size']}\n")
    f.write(f"Image Size: {TRAINING_PARAMS['image_size']}\n")
    f.write(f"Training Date: {timestamp}\n")
    f.write("\nModel saved to Google Drive at:\n")
    f.write(results_backup_path + "\n")

print(f"✓ Results backed up to Google Drive")
print(f"  Location: {results_backup_path}")
print(f"  - best.pt (trained model)")
print(f"  - inference_results/ (predictions)")
print(f"  - training_summary.txt (summary)")